In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import utils.base_utils as bu
import utils.window_utils as wu
from utils.macro_grouping import add_group_level, build_full_group_mapping, groups_as_array
from models.base import *
from models.classical import *
from models.ann_vector_validation import *
from models.linear import *
from models.tree import *

/Users/trineberntsensaether/anaconda3/envs/thesis_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Bianchi (1971-2018): Forward-only, revised data

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import seaborn as sns
import utils.base_utils as bu
import utils.window_utils as wu
import numpy as np
from utils.macro_grouping import add_group_level, build_full_group_mapping, groups_as_array

# Bianchi period:
start_date = '1971-08-31'
end_date = '2018-12-31'

# end_date = '2025-06-30' # kr and gsw end date
maturities = [str(i) for i in range(12, 121) if i % 12 == 0] # select only yearly maturities

yields = bu.get_yields(type='lw', start=start_date, end=end_date, maturities=maturities) # type can be kr, lw, gsw
forward = bu.get_forward_rates(yields)
xr = bu.get_excess_returns(yields, horizon=12).dropna() # horizon=12 means holding for 12 months

# adjust fred_md start_date by 6 months to fetch enough data for shifting
fred_md_start_date = pd.to_datetime(start_date) - pd.DateOffset(months=6)
fred_md_raw = bu.get_fred_data('data/2026-01-MD.csv', start=fred_md_start_date, end=end_date) # this is aligned to the last day of the previous month, so we get the same number of observations as the yields data

monthly_yields = bu.get_yields(type='lw', start=start_date, end=end_date, maturities=[str(i) for i in range(1, 121)]) # needed for monthly holding period excess returns. Not available for gsw
monthly_xr = bu.get_excess_returns(monthly_yields, horizon=1).dropna() # calculate monthly excess returns for robustness

# If wanted, apply per-series publication lag to latest-snapshot macro data
# from utils.publication_lags import apply_fred_md_publication_lag
# fred_md = apply_fred_md_publication_lag(fred_md_raw)  
# For results in paper, we naively shift all FRED-MD series by 1 month
# to reflect publication lag:
fred_md = fred_md_raw.shift(1) 

# Drop TWEXAFEGSMTHx and ACOGNO as they start late
fred_md = fred_md.drop(columns=['TWEXAFEGSMTHx', 'ACOGNO'])
# Finally, revert fred_md to start_date, after transformations and lag adjustments
fred_md = fred_md[start_date:end_date]

# Drop dates outside the xr range
yields = yields.loc[yields.index <= xr.index[-1]]
forward = forward.loc[forward.index <= xr.index[-1]]
xr = xr.loc[xr.index <= xr.index[-1]]
fred_md = fred_md.loc[fred_md.index <= xr.index[-1]]
monthly_xr = monthly_xr.loc[monthly_xr.index <= xr.index[-1]]

# Construct X with 3-level MultiIndex: (source, group, series)
s2g = build_full_group_mapping(fred_md, forward, yields)

X = pd.concat([fred_md, forward, yields],
               axis=1,
               keys=['fred', 'forward', 'yields'])

X = add_group_level(X, s2g, level_name='group')
X = X.sort_index(axis=1, level='group')
groups = groups_as_array(X, level='group')

y_all = xr[['24','36','48','60','72','84','96','108','120']].values
dates = xr.index

OOS_start = pd.Timestamp('1990-01-31')

/Users/trineberntsensaether/Documents/Master thesis/master_code/TIO4900-Replication/utils/macro_grouping.py:219: UserWarning: The following series are defined in get_fredmd_grouping() but are not present in the FRED-MD data: ['ACOGNO', 'TWEXAFEGSMTHx']. They may have been dropped or renamed.
  warnings.warn(
/Users/trineberntsensaether/Documents/Master thesis/master_code/TIO4900-Replication/utils/macro_grouping.py:168: UserWarning: 2 entries in series_to_group are not present in the DataFrame columns: ['ACOGNO', 'TWEXAFEGSMTHx']
  warnings.warn(


In [3]:
seeds = [1]
target_cols = ['24', '36', '48', '60', '84', '120']
targets = {
    '24': xr['24'],
    '36': xr['36'],
    '48': xr['48'],
    '60': xr['60'],
    '84': xr['84'],
    '120': xr['120'],
}

In [4]:
print(X.head())

                                           fred                                \
group      Consumption, Orders, and Inventories                                 
                                        AMDMNOx   AMDMUOx   ANDENOx   BUSINVx   
date                                                                            
1971-08-31                             0.054686  0.002978  0.124607  0.005666   
1971-09-30                             0.002027  0.008136  0.069380  0.004884   
1971-10-31                            -0.021259  0.002132 -0.110768  0.002620   
1971-11-30                             0.030169  0.003810  0.048664  0.000000   
1971-12-31                             0.024811  0.002878  0.044001  0.006388   

                                                                             \
group                                                               Housing   
           CMRMTSPLx DPCERA3M086SBEA  ISRATIOx   RETAILx UMCSENTx     HOUST   
date                             

In [5]:
cp_cols = ['24','36','48','60']

In [13]:
models = {
    #Table 1: Forwards ----------------------------------------
    #'PCA(3)': PCABaselineModel(components=3, series='forward'),
    #'PCA(5)': PCABaselineModel(components=5, series='forward'),
    #'PCA(10)': PCABaselineModel(components=10, series='forward'),

    #'Ridge': RidgeModel(series='forward'),
    #'Ridge_S': RidgeModelScaled(series='forward'),
    #'Lasso': LassoModel(series='forward'),
    #'Lasso_S': LassoModelScaled(series='forward'),

    # Table 2: Forwards & Macro --------------------------------
    #'PCA(8)': MacroPCA8Model(components=8, series='fred'),
    #'PCA L&N (2009)': LudvigsonNgModelNew(n_factors=8, series = 'fred'),
    #'Lasso w/macroLN and fwds': LassoMacroFwdDirectModel(macro_series='fred', forward_series='forward'),
    #'Ridge w/macroLN and fwds': RidgeMacroFwdDirectModel(macro_series='fred', forward_series='forward'),
    #'Lasso w/macroLN and CP factor': LassoMacroCPModel(xr_full = xr, cp_cols=['24','36','48','60'], macro_series='fred', forward_series='forward'),
    #'Ridge w/macroLN and CP factor': RidgeMacroCPModel(xr_full = xr, cp_cols=['24','36','48','60'], macro_series='fred', forward_series='forward'),
    #'Lasso w/macroFull and fwds': LassoRawMacroFwdDirectModel(macro_series='fred', forward_series='forward'),
    #'Ridge w/macroFull and fwds': RidgeRawMacroFwdDirectModel(macro_series='fred', forward_series='forward'),
    #'Lasso w/macroFull and fwds': LassoRawMacroFwdDirectModel(xr_full = xr, cp_cols=['24','36','48','60'], macro_series='fred', forward_series='forward'),
    #'Ridge w/macroFull and fwds': RidgeRawMacroFwdDirectModel(xr_full = xr, cp_cols=['24','36','48','60'], macro_series='fred', forward_series='forward'),
    'PCA on macro groups and fwds' : PCABaselineModelMacroGroups(series='forward'),
    'PCA on macro groups and fwds with Lasso' : PCABaselineModelMacroGroups(series='forward', lasso = True),

}

In [14]:
forward.columns

Index(['12', '24', '36', '48', '60', '72', '84', '96', '108', '120'], dtype='object')

In [15]:
results = []
forecast_store = {}

In [16]:
for seed in seeds:
    print(f'Running seed {seed}...')

    for model_name, model in models.items():
        print(f'\nModel: {model_name}')

        for target_name, y in targets.items():
            print(f'  Target: {target_name}')

            y_forecast = wu.expanding_window(
                model,
                X,
                y,
                dates,
                OOS_start,
                gap=0,
                refit_freq=1,
                coef_callback=None
            )

            # store results
            forecast_store.setdefault(target_name, {})
            forecast_store[target_name][model_name] = y_forecast

            # compute metrics
            r2_hist = wu.oos_r2(y, y_forecast, benchmark='hist_mean')
            pval_hist = bu.RSZ_Signif(y, y_forecast)

            results.append({
                'seed': seed,
                'target': target_name,
                'model': model_name,
                'r2_hist_mean': r2_hist,
                'pval_hist_mean': pval_hist
            })

Running seed 1...

Model: PCA on macro groups and fwds
  Target: 24


/Users/trineberntsensaether/Documents/Master thesis/master_code/TIO4900-Replication/utils/base_utils.py:456: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return 1-tstat.cdf(results.tvalues[0], results.nobs-1)


  Target: 36


/Users/trineberntsensaether/Documents/Master thesis/master_code/TIO4900-Replication/utils/base_utils.py:456: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return 1-tstat.cdf(results.tvalues[0], results.nobs-1)


  Target: 48


...

/Users/trineberntsensaether/Documents/Master thesis/master_code/TIO4900-Replication/utils/base_utils.py:456: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return 1-tstat.cdf(results.tvalues[0], results.nobs-1)


  Target: 84


/Users/trineberntsensaether/Documents/Master thesis/master_code/TIO4900-Replication/utils/base_utils.py:456: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return 1-tstat.cdf(results.tvalues[0], results.nobs-1)


  Target: 120


/Users/trineberntsensaether/Documents/Master thesis/master_code/TIO4900-Replication/utils/base_utils.py:456: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return 1-tstat.cdf(results.tvalues[0], results.nobs-1)


In [17]:
X.index.equals(xr.index)

True

In [18]:
forward.columns

Index(['12', '24', '36', '48', '60', '72', '84', '96', '108', '120'], dtype='object')

In [19]:
results_df = pd.DataFrame(results)

results_df_display = results_df.copy()

results_df_display['pval_hist_mean'] = results_df_display.apply(
    lambda row: row['pval_hist_mean'] if row['r2_hist_mean'] > 0 else np.nan,
    axis=1
)

results_df_display['r2_hist_mean'] = results_df_display['r2_hist_mean'] * 100

results_df_display = results_df_display.round({
    'r2_hist_mean': 1,
    'pval_hist_mean': 4
})

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
print(results_df_display)


    seed target                                    model  r2_hist_mean  \
0      1     24             PCA on macro groups and fwds          -2.2   
1      1     36             PCA on macro groups and fwds           0.8   
2      1     48             PCA on macro groups and fwds           0.3   
3      1     60             PCA on macro groups and fwds           2.6   
4      1     84             PCA on macro groups and fwds           4.2   
5      1    120             PCA on macro groups and fwds           8.1   
6      1     24  PCA on macro groups and fwds with Lasso          -0.0   
7      1     36  PCA on macro groups and fwds with Lasso           4.1   
8      1     48  PCA on macro groups and fwds with Lasso          10.0   
9      1     60  PCA on macro groups and fwds with Lasso          13.4   
10     1     84  PCA on macro groups and fwds with Lasso          16.1   
11     1    120  PCA on macro groups and fwds with Lasso          18.3   

    pval_hist_mean  
0              N

## Bianchi (1971-2018): Forward + macro, revised data

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import seaborn as sns
import utils.base_utils as bu
import utils.window_utils as wu
import numpy as np
from utils.macro_grouping import add_group_level, build_full_group_mapping, groups_as_array

# Bianchi period:
start_date = '1971-08-31'
end_date = '2018-12-31'

# end_date = '2025-06-30' # kr and gsw end date
maturities = [str(i) for i in range(12, 121) if i % 12 == 0] # select only yearly maturities

yields = bu.get_yields(type='lw', start=start_date, end=end_date, maturities=maturities) # type can be kr, lw, gsw
forward = bu.get_forward_rates(yields)
xr = bu.get_excess_returns(yields, horizon=12).dropna() # horizon=12 means holding for 12 months

# adjust fred_md start_date by 6 months to fetch enough data for shifting
fred_md_start_date = pd.to_datetime(start_date) - pd.DateOffset(months=6)
fred_md_raw = bu.get_fred_data('data/2026-01-MD.csv', start=fred_md_start_date, end=end_date) # this is aligned to the last day of the previous month, so we get the same number of observations as the yields data

monthly_yields = bu.get_yields(type='lw', start=start_date, end=end_date, maturities=[str(i) for i in range(1, 121)]) # needed for monthly holding period excess returns. Not available for gsw
monthly_xr = bu.get_excess_returns(monthly_yields, horizon=1).dropna() # calculate monthly excess returns for robustness

# If wanted, apply per-series publication lag to latest-snapshot macro data
# from utils.publication_lags import apply_fred_md_publication_lag
# fred_md = apply_fred_md_publication_lag(fred_md_raw)  
# For results in paper, we naively shift all FRED-MD series by 1 month
# to reflect publication lag:
fred_md = fred_md_raw.shift(1) 

# Drop TWEXAFEGSMTHx and ACOGNO as they start late
fred_md = fred_md.drop(columns=['TWEXAFEGSMTHx', 'ACOGNO'])
# Finally, revert fred_md to start_date, after transformations and lag adjustments
fred_md = fred_md[start_date:end_date]

# Drop dates outside the xr range
yields = yields.loc[yields.index <= xr.index[-1]]
forward = forward.loc[forward.index <= xr.index[-1]]
xr = xr.loc[xr.index <= xr.index[-1]]
fred_md = fred_md.loc[fred_md.index <= xr.index[-1]]
monthly_xr = monthly_xr.loc[monthly_xr.index <= xr.index[-1]]

# Construct X with 3-level MultiIndex: (source, group, series)
s2g = build_full_group_mapping(fred_md, forward, yields)

X = pd.concat([fred_md, forward, yields],
               axis=1,
               keys=['fred', 'forward', 'yields'])

X = add_group_level(X, s2g, level_name='group')
X = X.sort_index(axis=1, level='group')
groups = groups_as_array(X, level='group')

y_all = xr[['24','36','48','60','72','84','96','108','120']].values
dates = xr.index

OOS_start = pd.Timestamp('1990-01-31')

In [ ]:
seeds = [1]

In [ ]:
targets = {
    '24': xr['24'],
    '36': xr['36'],
    '48': xr['48'],
    '60': xr['60'],
    '84': xr['84'],
    '120': xr['120'],
}

In [ ]:
print(fred_md.head())

In [ ]:
models = {

    'PCA(8)': MacroPCA8Model(components=8, series='fred'),
    'PCA L&N (2009)': LudvigsonNgModelNew(n_factors=8, series = 'fred'),

}

In [ ]:
results = []
forecast_store = {}

In [ ]:
for seed in seeds:
    print(f'Running seed {seed}...')

    for model_name, model in models.items():
        print(f'\nModel: {model_name}')

        for target_name, y in targets.items():
            print(f'  Target: {target_name}')

            y_forecast = wu.expanding_window(
                model,
                X,
                y,
                dates,
                OOS_start,
                gap=0,
                refit_freq=1,
                coef_callback=None
            )

            # store results
            forecast_store.setdefault(target_name, {})
            forecast_store[target_name][model_name] = y_forecast

            # compute metrics
            r2_hist = wu.oos_r2(y, y_forecast, benchmark='hist_mean')
            pval_hist = bu.RSZ_Signif(y, y_forecast)

            results.append({
                'seed': seed,
                'target': target_name,
                'model': model_name,
                'r2_hist_mean': r2_hist,
                'pval_hist_mean': pval_hist
            })

            print(f'R2: {r2_hist}')
            print(f'p-value: {pval_hist}')

## Bianchi (1971-2018): Forward + macro, non-revised data

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import seaborn as sns
import utils.base_utils as bu
import utils.window_utils as wu
import numpy as np
from utils.macro_grouping import add_group_level, build_full_group_mapping, groups_as_array

# Bianchi period:
start_date = '1971-08-31'
end_date = '2018-12-31'

# end_date = '2025-06-30' # kr and gsw end date
maturities = [str(i) for i in range(12, 121) if i % 12 == 0]  # yearly maturities

yields = bu.get_yields(type='lw', start=start_date, end=end_date, maturities=maturities)
forward = bu.get_forward_rates(yields)
xr = bu.get_excess_returns(yields, horizon=12).dropna()

monthly_yields = bu.get_yields(
    type='lw',
    start=start_date,
    end=end_date,
    maturities=[str(i) for i in range(1, 121)]
)
monthly_xr = bu.get_excess_returns(monthly_yields, horizon=1).dropna()

# Load transformed real-time macro data
realtime_tcode_balanced_df = pd.read_csv("../data/ALFRED/simple_outputs/realtime_tcode_balanced.csv")

# Set date index
realtime_tcode_balanced_df["decision_date"] = pd.to_datetime(
    realtime_tcode_balanced_df["decision_date"]
)
realtime_tcode_balanced_df = realtime_tcode_balanced_df.set_index("decision_date").sort_index()

# Apply the same one-month lag logic as befor

# Drop series that start late
fred = fred.drop(columns=['TWEXAFEGSMTHx', 'ACOGNO'], errors='ignore')

# Restrict sample after lagging
fred = fred.loc[start_date:end_date]

# Drop dates outside the xr range
yields = yields.loc[yields.index <= xr.index[-1]]
forward = forward.loc[forward.index <= xr.index[-1]]
xr = xr.loc[xr.index <= xr.index[-1]]
fred = fred.loc[fred.index <= xr.index[-1]]
monthly_xr = monthly_xr.loc[monthly_xr.index <= xr.index[-1]]

# Optional but often useful: force common index across predictors/target
common_idx = xr.index.intersection(yields.index).intersection(forward.index).intersection(fred.index)
yields = yields.loc[common_idx]
forward = forward.loc[common_idx]
xr = xr.loc[common_idx]
fred = fred.loc[common_idx]
monthly_xr = monthly_xr.loc[monthly_xr.index <= common_idx[-1]]

# Construct X with 3-level MultiIndex: (source, group, series)
s2g = build_full_group_mapping(fred, forward, yields)

X = pd.concat(
    [fred, forward, yields],
    axis=1,
    keys=['fred', 'forward', 'yields']
)

X = add_group_level(X, s2g, level_name='group')
X = X.sort_index(axis=1, level='group')
groups = groups_as_array(X, level='group')

y_all = xr[['24', '36', '48', '60', '72', '84', '96', '108', '120']].values
dates = xr.index

OOS_start = pd.Timestamp('1990-01-31')

In [ ]:
print(X.isna().sum().sum())          # total number of NaNs
print(X.isna().mean().mean())        # fraction of missing values

In [ ]:
for block in ['fred', 'forward', 'yields']:
    print(f"\n--- {block} ---")
    
    df_block = X[block]
    nan_cols = df_block.columns[df_block.isna().any()].tolist()
    
    print("Total NaNs:", df_block.isna().sum().sum())
    print("Columns with NaNs:", nan_cols)
    
    if nan_cols:
        # Drop using full MultiIndex keys
        cols_to_drop = [(block, col) for col in nan_cols]
        X = X.drop(columns=cols_to_drop)
        print(f"Dropped {len(nan_cols)} columns from {block}")

In [ ]:
for block in ['fred', 'forward', 'yields']:
    print(f"\n--- {block} ---")
    
    df_block = X[block]
    nan_cols = df_block.columns[df_block.isna().any()].tolist()
    
    print("Total NaNs:", df_block.isna().sum().sum())
    print("Columns with NaNs:", nan_cols)
    
    if nan_cols:
        cols_to_keep = []
        for col in X.columns:
            if col[0] == block and col[1:] in nan_cols:
                continue
            cols_to_keep.append(col)
        
        X = X.loc[:, cols_to_keep]
        print(f"Dropped {len(nan_cols)} columns from {block}")

In [ ]:
nan_cols = fred.columns[fred.isna().any()]
fred = fred.drop(columns=nan_cols)

In [ ]:
for block in ['fred', 'forward', 'yields']:
    print(f"\n--- {block} ---")
    df_block = X[block]
    print("Total NaNs:", df_block.isna().sum().sum())
    print("Columns with NaNs:", df_block.columns[df_block.isna().any()].tolist())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.imshow(X.isna(), aspect='auto', interpolation='none')
plt.title("NaN pattern in X (rows=time, cols=variables)")
plt.xlabel("Variables")
plt.ylabel("Time")
plt.colorbar()
plt.show()

In [ ]:
print(X.head())

In [ ]:
cp_cols = ['24', '36', '48', '60']

models = {
    'PCA as in Ludvigson and Ng (2009)': LudvigsonNgWithCPModel(
        xr_full=xr,
        cp_cols=cp_cols,
        n_factors=8
    ),
    'PCA - first 8 PCs': PCAFirst8PCsWithCPModel(xr_full=xr, cp_cols=cp_cols, components=8),

}

In [ ]:
for seed in seeds:
    print(f'Running seed {seed}...')

    for model_name, model in models.items():
        print(f'\nModel: {model_name}')

        for target_name, y in targets.items():
            print(f'  Target: {target_name}')

            y_forecast = wu.expanding_window(
                model,
                X,
                y,
                dates,
                OOS_start,
                gap=0,
                refit_freq=1,
                coef_callback=None
            )

            # store results
            forecast_store.setdefault(target_name, {})
            forecast_store[target_name][model_name] = y_forecast

            # compute metrics
            r2_hist = wu.oos_r2(y, y_forecast, benchmark='hist_mean')
            pval_hist = bu.RSZ_Signif(y, y_forecast)

            results.append({
                'seed': seed,
                'target': target_name,
                'model': model_name,
                'r2_hist_mean': r2_hist,
                'pval_hist_mean': pval_hist
            })

            print(f'R2: {r2_hist}')
            print(f'p-value: {pval_hist}')